# Applied Machine Learning (CSAI2017P) — Lab Assignment 8
## Anomaly Detection and Physiological Signal Classification

Experiments 10–11 · CO2, CO3, CO4 · **10 marks**

| | |
|---|---|
| Name | Mansha |
| SAP ID | 590021940 |
| Batch | 06 |
| Due | announced in the lab |
| Lab quiz | LQ8, after submission |

**Datasets:** A2 (Telco), C3 (UCI Human Activity Recognition)

---

### Before you start
- Attempt **2 of 3** in Section A, **1 of 2** in Section B. Section C is compulsory.
- Fit every transformer inside a `Pipeline`. Never fit on the test set.
- Set `random_state` wherever the question asks for reproducibility.
- Every question wants a short written observation, not just a number.
- Restart the kernel and run top-to-bottom before you submit.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

for m in (np, pd, sklearn):
    print(f"{m.__name__:12s} {m.__version__}")


numpy        2.1.3
pandas       2.2.3
sklearn      1.6.1


## Load the data

Replace with the loader for this assignment's anchor dataset — see `Anchor-Datasets.pdf` for the snippets.

In [2]:
# TODO: load the dataset for this assignment
# from sklearn.datasets import ...
# df = pd.read_csv('data/...')
# Load both datasets used in this assignment

import os
import zipfile
import urllib.request
import kagglehub

# A2 — Telco Customer Churn


telco_path = kagglehub.dataset_download(
    "blastchar/telco-customer-churn"
)

telco_file = os.path.join(
    telco_path,
    "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df = pd.read_csv(telco_file)

# TotalCharges contains blank strings, so convert safely to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print("A2 Telco shape:", df.shape)


# C3 — UCI Human Activity Recognition

har_url = (
    "https://archive.ics.uci.edu/static/public/240/"
    "human+activity+recognition+using+smartphones.zip"
)

zip_file = "har.zip"
extract_dir = "har_data"

# Download only if it is not already present
if not os.path.exists(zip_file):
    urllib.request.urlretrieve(har_url, zip_file)

# Extract only if necessary
if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_file, "r") as z:
        z.extractall(extract_dir)

base = os.path.join(extract_dir, "UCI HAR Dataset")

X_train = pd.read_csv(
    os.path.join(base, "train", "X_train.txt"),
    sep=r"\s+",
    header=None
)

y_train = pd.read_csv(
    os.path.join(base, "train", "y_train.txt"),
    sep=r"\s+",
    header=None
).iloc[:, 0]

X_test = pd.read_csv(
    os.path.join(base, "test", "X_test.txt"),
    sep=r"\s+",
    header=None
)

y_test = pd.read_csv(
    os.path.join(base, "test", "y_test.txt"),
    sep=r"\s+",
    header=None
).iloc[:, 0]

print("C3 HAR train shape:", X_train.shape)
print("C3 HAR test shape:", X_test.shape)

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
A2 Telco shape: (7043, 21)


FileNotFoundError: [Errno 2] No such file or directory: 'har_data/UCI HAR Dataset/train/X_train.txt'

---

# Section A — attempt any 2 of 3 (2 marks each)

*attempt 2 of 3*

## A1. Statistical outliers  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2, numeric columns.*

a. Flag outliers in `tenure`, `MonthlyCharges` and `TotalCharges` using the IQR rule and the 3σ rule.
b. Report how many rows each rule flags and how many they agree on.
c. State in two lines which rule you would trust on a skewed column, and why.


In [ ]:
# A1 — code

cols = ["tenure", "MonthlyCharges", "TotalCharges"]
X_a1 = df[cols].dropna()
Q1 = X_a1.quantile(0.25)
Q3 = X_a1.quantile(0.75)
IQR = Q3 - Q1

iqr_outliers = (
    (X_a1 < (Q1 - 1.5 * IQR)) |
    (X_a1 > (Q3 + 1.5 * IQR))
)
iqr_flag = iqr_outliers.any(axis=1)

mean = X_a1.mean()
std = X_a1.std()

sigma_outliers = (
    (X_a1 < (mean - 3 * std)) |
    (X_a1 > (mean + 3 * std))
)

sigma_flag = sigma_outliers.any(axis=1)
both_flag = iqr_flag & sigma_flag

print("Rows analysed:", len(X_a1))
print("Rows flagged by IQR rule:", iqr_flag.sum())
print("Rows flagged by 3-sigma rule:", sigma_flag.sum())
print("Rows flagged by both rules:", both_flag.sum())

print("\nOutliers per column:")
print(pd.DataFrame({
    "IQR": iqr_outliers.sum(),
    "3-sigma": sigma_outliers.sum()
}))


**Observation (A1):**

> The IQR and 3σ rules may flag different numbers of rows because they use different criteria for identifying extreme values. For a skewed column, I would trust the IQR rule more because it is based on quartiles and is less affected by extreme values, whereas the 3σ rule is more suitable for approximately normal data.

## A2. Isolation Forest  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2.*

a. Fit `IsolationForest` with `contamination=0.05` on the scaled numeric features.
b. Report how many rows are flagged and inspect five of them.
c. State in two lines what makes those rows unusual.


In [ ]:
# A2 — code

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

X_a2 = df[numeric_features].dropna()
iso_model = Pipeline([
    ("scaler", StandardScaler()),
    ("isolation_forest", IsolationForest(
        contamination=0.05,
        random_state=RANDOM_STATE
    ))
])

pred_a2 = iso_model.fit_predict(X_a2)
anomaly_flag = pred_a2 == -1

print("Rows analysed:", len(X_a2))
print("Rows flagged as anomalies:", anomaly_flag.sum())

flagged_indices = X_a2.index[anomaly_flag]

print("\nFive flagged rows:")
display(
    df.loc[
        flagged_indices,
        ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]
    ].head(5)
)

**Observation (A2):**

>Isolation Forest flags approximately 5% of the observations, consistent with the specified contamination rate. The inspected records contain unusual combinations of tenure and charge values relative to typical customers. Such observations are easier to isolate using random partitions, causing the model to classify them as anomalies.

## A3. Load the signal features  &nbsp;&nbsp;`[2 marks]`

*Dataset: C3 (HAR).*

a. Load the pre-computed 561-feature train and test sets and report both shapes.
b. Report the class distribution across the six activities.
c. State in two lines why the authors supplied their own split rather than letting you make one.


In [ ]:
# A3 — code


**Observation (A3):**

> *write 2–6 lines here*

---

# Section B — attempt any 1 of 2 (4 marks each)

*attempt 1 of 2*

## B1. Subject-wise versus random split  &nbsp;&nbsp;`[4 marks]`

*Dataset: C3 (HAR).*

a. Train a linear SVM using the supplied subject-wise split and report test accuracy.
b. Pool train and test, reshuffle randomly at the same ratio, retrain and report accuracy again.
c. Explain in 6–8 lines why the random figure is inflated and what it would cost you in deployment.


In [ ]:
# B1 — code

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

subject_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        random_state=RANDOM_STATE,
        max_iter=10000
    ))
])

subject_model.fit(X_train, y_train)

y_pred_subject = subject_model.predict(X_test)

subject_accuracy = accuracy_score(
    y_test,
    y_pred_subject
)

print(
    f"Subject-wise test accuracy: "
    f"{subject_accuracy * 100:.2f}%"
)

X_all = pd.concat(
    [X_train, X_test],
    ignore_index=True
)

y_all = pd.concat(
    [y_train, y_test],
    ignore_index=True
)

test_ratio = len(X_test) / len(X_all)

X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X_all,
        y_all,
        test_size=test_ratio,
        random_state=RANDOM_STATE,
        stratify=y_all
    )
)

random_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        random_state=RANDOM_STATE,
        max_iter=10000
    ))
])

random_model.fit(
    X_train_random,
    y_train_random
)

y_pred_random = random_model.predict(
    X_test_random
)

random_accuracy = accuracy_score(
    y_test_random,
    y_pred_random
)
print(
    f"Random-split accuracy: "
    f"{random_accuracy * 100:.2f}%"
)

print(
    f"Difference: "
    f"{(random_accuracy - subject_accuracy) * 100:.2f} "
    f"percentage points"
)

**Observation (B1):**

> The random split gives a higher accuracy because samples from the same person can appear in both the training and test sets. Each person has characteristic movement patterns, so the model may learn some subject-specific information in addition to the actual activity patterns. This makes the randomly created test set less independent and can inflate the measured accuracy. In the supplied subject-wise split, the people in the test set are unseen during training, giving a more realistic evaluation. In deployment, the classifier will often have to recognise activities performed by new users. Therefore, the random-split accuracy may overestimate the performance that would be obtained in real-world use.

## B2. Anomalies without labels  &nbsp;&nbsp;`[4 marks]`

*Dataset: A2 or C3, your choice.*

a. Run Isolation Forest, One-Class SVM and Local Outlier Factor at the same contamination rate.
b. Report how many points all three agree on, and how many only one method flags.
c. In 6–8 lines, explain how you would decide which method is right when you have no labels.


In [ ]:
# B2 — code


**Observation (B2):**

> *write 2–6 lines here*

---

# Section C — compulsory (2 marks)

*compulsory*

## C1. Which activities blur  &nbsp;&nbsp;`[2 marks]`

*Dataset: C3 (HAR).*

a. Plot the confusion matrix from your subject-wise model.
b. Name the two activities most often confused and explain in 4–6 lines why their sensor signatures resemble each other.


In [ ]:
# C1 — code

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

activity_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

labels = [1, 2, 3, 4, 5, 6]
cm = confusion_matrix(
    y_test,
    y_pred_subject,
    labels=labels
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=activity_names
)

fig, ax = plt.subplots(figsize=(9, 7))

disp.plot(
    ax=ax,
    xticks_rotation=45,
    colorbar=False
)

plt.title("Confusion Matrix — Subject-wise Linear SVM")
plt.tight_layout()
plt.show()

pair_confusions = []

for i in range(len(activity_names)):
    for j in range(i + 1, len(activity_names)):

        total_confusion = cm[i, j] + cm[j, i]

        pair_confusions.append(
            (
                total_confusion,
                activity_names[i],
                activity_names[j]
            )
        )

pair_confusions.sort(reverse=True)

most_confused = pair_confusions[0]

print(
    "Most confused pair:",
    most_confused[1],
    "and",
    most_confused[2]
)

print(
    "Total mutual confusions:",
    most_confused[0]
)

**Observation (C1):**

> The activities most often confused are SITTING and STANDING. Both are static activities with very little body movement, so their accelerometer and gyroscope signals have similar patterns. Unlike walking activities, they do not produce strong repetitive motion signals. Their main difference is body posture rather than movement, making their sensor features harder to separate. This similarity causes some sitting and standing samples to be misclassified as each other.

---

## Self-check before submitting

- [ ] Kernel restarted and run top-to-bottom, all outputs visible
- [ ] Every attempted question has a written observation
- [ ] No transformer fitted outside a pipeline
- [ ] Metrics reported with units where they have any
- [ ] File named `AML_LA08_<SAPID>.ipynb`
